# 02 — Baseline vs RAG Query Comparison

**Issue #8** — Compare answers from a plain LLM (no retrieval) against the full RAG pipeline using the real ECB speeches corpus.

## Prerequisites

1. Ollama running: `ollama serve`
2. ECB speeches CSV at `inputs/Data/all_ECB_speeches_csv.csv`
3. Environment ready: `python scripts/ollama_ready.py` (ingests corpus into ChromaDB if not already done)

## What this notebook shows

| Mode | How it works | Expected behaviour |
|------|-------------|--------------------|
| **Baseline** | Question sent directly to LLM — no retrieval | General answer from training data; no citations |
| **RAG** | Top-4 ECB speech chunks retrieved first, then injected into prompt | Answer grounded in specific speeches; source metadata returned |

In [ ]:
import sys, os
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv("../.env")

import requests

MODEL     = os.getenv("OLLAMA_MODEL", "llama3.2")
CHROMA_DIR = "../chroma_db"

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    available = [m["name"] for m in r.json().get("models", [])]
    print(f"Ollama is running.")
    print(f"  Model in use  : {MODEL}")
    print(f"  Models on disk: {available}")
except Exception:
    raise RuntimeError(
        "Ollama is not running.\n"
        "Start it with: ollama serve\n"
        "Then run:      python scripts/ollama_ready.py"
    )

## 1 — Load vectorstore and build chains

In [ ]:
from rag.query import load_vectorstore, build_rag_chain
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Load the ChromaDB collection created by ingest_to_chromadb
vectorstore = load_vectorstore(persist_dir=CHROMA_DIR)

# RAG chain: retriever -> prompt -> LLM
rag_chain = build_rag_chain(vectorstore, model=MODEL, k=4)

# Baseline: raw LLM call with no retrieved context
llm = ChatOllama(model=MODEL)

def ask_baseline(question: str) -> str:
    """Send question directly to LLM — no retrieval, no context injection."""
    response = llm.invoke([HumanMessage(content=question)])
    return response.content

print("Vectorstore and chains ready.")

## 2 — Test questions

Three questions chosen to exercise different aspects of ECB monetary policy — topics the corpus should cover well.

In [ ]:
QUESTIONS = [
    "What tools does the ECB use to control inflation?",
    "How did the ECB respond to rising inflation in 2022?",
    "What is the ECB's forward guidance strategy?",
]

for i, q in enumerate(QUESTIONS, 1):
    print(f"Q{i}: {q}")

## 3 — Baseline answers (no retrieval)

In [ ]:
baseline_results = []

for i, question in enumerate(QUESTIONS, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {question}")
    print(f"{'='*60}")
    answer = ask_baseline(question)
    baseline_results.append({"question": question, "answer": answer})
    print(answer)

## 4 — RAG answers (with retrieval from ECB speeches)

In [ ]:
from rag.query import ask

rag_results = []

for i, question in enumerate(QUESTIONS, 1):
    print(f"\n{'='*60}")
    print(f"Q{i}: {question}")
    print(f"{'='*60}")
    result = ask(rag_chain, question)
    rag_results.append({"question": question, **result})
    print(result["answer"])
    print(f"\nSources ({len(result['sources'])} chunks retrieved):")
    for s in result["sources"]:
        print(f"  - {s.get('speakers', 'Unknown')} | {s.get('date', '?')} | {s.get('title', '?')}")

## 5 — Side-by-side comparison

In [ ]:
from IPython.display import HTML, display

rows = ""
for i, (b, r) in enumerate(zip(baseline_results, rag_results), 1):
    rows += (
        f"<tr style='border-top:2px solid #dee2e6'>"
        f"<td style='padding:8px;font-weight:bold;vertical-align:top' colspan='2'>Q{i}: {b['question']}</td>"
        f"</tr>"
        f"<tr>"
        f"<td style='padding:8px;vertical-align:top;width:50%;background:#fff8f0'>"
        f"<strong>Baseline</strong><br><small style='color:#888'>No retrieval — LLM training data only</small><br><br>"
        f"{b['answer'].replace(chr(10), '<br>')}</td>"
        f"<td style='padding:8px;vertical-align:top;width:50%;background:#f0fff4'>"
        f"<strong>RAG</strong><br><small style='color:#888'>Top-4 ECB speech chunks retrieved</small><br><br>"
        f"{r['answer'].replace(chr(10), '<br>')}<br><br>"
        f"<small><em>Sources: " +
        ", ".join(f"{s.get('speakers','?')} ({s.get('date','?')[:4]})" for s in r['sources']) +
        "</em></small></td>"
        f"</tr>"
    )

display(HTML(
    "<table style='width:100%;border-collapse:collapse;font-size:0.9em'>"
    "<thead><tr style='background:#343a40;color:white'>"
    "<th style='padding:8px;width:50%'>Baseline (no retrieval)</th>"
    "<th style='padding:8px;width:50%'>RAG (ECB corpus retrieval)</th>"
    "</tr></thead><tbody>" + rows + "</tbody></table>"
))

## 6 — Observations

Fill in after running the cells above. Consider:

- Where does RAG improve on baseline? (specificity, citation, grounding)
- Where does baseline do just as well? (well-known facts already in LLM training data)
- Are the retrieved sources relevant to the question asked?
- Any cases where retrieval introduces noise or contradicts the LLM's training?

These observations will feed directly into the Phase 3 evaluation harness (issues #9–#12).